# Credit Card Fraud Detection — EDA & Modeling

This notebook walks through the complete exploratory data analysis and modeling pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/creditcard.csv')
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df.Class.mean()*100:.3f}%')
df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(x='Class', data=df, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])

# Log-scale
counts = df['Class'].value_counts()
axes[1].bar(['Legitimate', 'Fraud'], [counts[0], counts[1]], color=['#2ecc71', '#e74c3c'])
axes[1].set_yscale('log')
axes[1].set_title('Class Distribution (Log Scale)')
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, cls in enumerate([0, 1]):
    label = 'Legitimate' if cls == 0 else 'Fraud'
    color = '#2ecc71' if cls == 0 else '#e74c3c'
    axes[idx].hist(df[df.Class == cls]['Amount'], bins=50, color=color, alpha=0.7)
    axes[idx].set_title(f'{label} — Amount Distribution')
    axes[idx].set_xlabel('Amount')
    axes[idx].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — top features correlated with Class
correlations = df.corr()['Class'].drop('Class').sort_values()
top_features = pd.concat([correlations.head(10), correlations.tail(10)])

plt.figure(figsize=(10, 8))
top_features.plot(kind='barh', color=['#e74c3c' if v < 0 else '#2ecc71' for v in top_features])
plt.title('Top Features Correlated with Fraud')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Run the full training pipeline
# (Alternatively, run from terminal: python -m src.train)

import sys
sys.path.insert(0, '..')

from src.data_preparation import load_data, split_data, preprocess
from src.feature_engineering import engineer_features
from src.model import apply_resampling, optimize_hyperparameters, train_model, find_optimal_threshold, evaluate_model
from src.explainability import generate_shap_explanations

# Load and prepare
df = load_data()
X_train, X_test, y_train, y_test = split_data(df)
X_train, X_test, scaler = preprocess(X_train, X_test)
X_train = engineer_features(X_train)
X_test = engineer_features(X_test)

# Resample
X_train_res, y_train_res = apply_resampling(X_train, y_train)

# Optimize
best_params, study = optimize_hyperparameters(X_train_res, y_train_res)

# Train
model = train_model(X_train_res, y_train_res, best_params)

# Evaluate
y_prob = model.predict_proba(X_test)[:, 1]
threshold = find_optimal_threshold(y_test, y_prob)
metrics, y_prob, y_pred, cm = evaluate_model(model, X_test, y_test, threshold)

In [ ]:
# SHAP explanations
shap_values = generate_shap_explanations(model, X_test.iloc[:500])